# Notebook 2: Dataset Perparation

**Goal:** Scan chunks 002–008, run Simple RAG and HyperRAG on every question, and Prepare subset — questions where:

- Simple RAG **missed** the gold answer (EM = 0)
- HyperRAG **hit** the gold answer via graph expansion (EM = 1)

These are the cases where the 1-hop hyperlink graph step provably matters.

**Output files** (saved to `data/`):
- `corpus-graph-wins.json` — Wikipedia pages required to answer graph-wins questions
- `questions-graph-wins.json` — the graph-wins questions with their metadata

**Prerequisite:** Chunks 002–008 must exist in `data/` (created by `01_data_exploration.ipynb`).

In [ ]:
import json
import html as html_lib
import gc
from pathlib import Path
from collections import Counter

import faiss
import numpy as np
import networkx as nx
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
if Path('utils.py').exists():
    BASE_DIR = Path('.')
elif Path('notebooks/utils.py').exists():
    BASE_DIR = Path('notebooks')
else:
    raise FileNotFoundError('Run from notebooks/ or from HyperRAG/')

DATA_DIR = BASE_DIR / 'data'

# Chunks to process (001 already has inference done)
CHUNKS_TO_PROCESS = [f'{i:03d}' for i in range(2, 9)]  # 002 … 008

# Output paths
OUT_CORPUS    = DATA_DIR / 'corpus-graph-wins.json'
OUT_QUESTIONS = DATA_DIR / 'questions-graph-wins.json'

# HyperRAG hyper-parameters (keep identical to notebook 04)
K        = 5   # FAISS top-k pages
EXPAND_K = 5   # graph expansion pages

print(f'Chunks to process : {CHUNKS_TO_PROCESS}')
print(f'Output corpus     : {OUT_CORPUS}')
print(f'Output questions  : {OUT_QUESTIONS}')

W0504 09:24:31.454000 52412 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Chunks to process : ['002', '003', '004', '005', '006', '007', '008']
Output corpus     : data\corpus-graph-wins.json
Output questions  : data\questions-graph-wins.json


---
## Helper Functions

All functions below are identical to the implementations in notebooks 02 and 04 — reproduced
here so this notebook is self-contained.

In [2]:
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
_model = None

def get_model() -> SentenceTransformer:
    """Load (or return cached) SentenceTransformer model."""
    global _model
    if _model is None:
        print(f'Loading {MODEL_NAME} ...')
        _model = SentenceTransformer(MODEL_NAME)
        print('Model ready.')
    return _model


def embed_texts(texts: list[str]) -> np.ndarray:
    """Embed texts into L2-normalised float32 vectors (shape: N × 384)."""
    model = get_model()
    vecs  = model.encode(texts, show_progress_bar=len(texts) > 100, convert_to_numpy=True)
    vecs  = vecs.astype('float32')
    faiss.normalize_L2(vecs)
    return vecs


def build_index(corpus: list[dict]) -> faiss.IndexFlatIP:
    """Embed all corpus pages and store in a FAISS inner-product index."""
    vecs  = embed_texts([p['text'] for p in corpus])
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    return index


def retrieve(
    query:  str,
    index:  faiss.IndexFlatIP,
    corpus: list[dict],
    k:      int = 5,
) -> list[dict]:
    """Return top-k corpus pages most similar to query."""
    q_vec = embed_texts([query])
    scores, indices = index.search(q_vec, k)
    results = []
    for idx, score in zip(indices[0], scores[0]):
        if 0 <= idx < len(corpus):
            page = dict(corpus[idx])
            page['_score'] = float(score)
            results.append(page)
    return results


def build_graph(corpus: list[dict]) -> nx.DiGraph:
    """Build directed hyperlink graph — edges only between corpus pages."""
    corpus_titles = {p['title'] for p in corpus}
    G = nx.DiGraph()
    G.add_nodes_from(corpus_titles)
    for page in corpus:
        src = page['title']
        for link in page.get('links', []):
            if link in corpus_titles and link != src:
                G.add_edge(src, link)
    return G


def expand_neighbors(
    query:         str,
    initial_pages: list[dict],
    corpus:        list[dict],
    graph:         nx.DiGraph,
    expand_k:      int = 3,
) -> list[dict]:
    """1-hop graph expansion: find neighbors of initial pages and re-rank by query similarity."""
    title_to_page   = {p['title']: p for p in corpus}
    initial_titles  = {p['title'] for p in initial_pages}
    neighbor_titles = set()

    for page in initial_pages:
        if not graph.has_node(page['title']):
            continue
        for nbr in graph.successors(page['title']):
            if nbr in title_to_page and nbr not in initial_titles:
                neighbor_titles.add(nbr)
        for nbr in graph.predecessors(page['title']):
            if nbr in title_to_page and nbr not in initial_titles:
                neighbor_titles.add(nbr)

    if not neighbor_titles or expand_k <= 0:
        return []

    neighbor_pages = [title_to_page[t] for t in neighbor_titles]
    n_embed = embed_texts([p['text'] for p in neighbor_pages])
    q_embed = embed_texts([query])
    scores  = (q_embed @ n_embed.T)[0]
    top_idx = scores.argsort()[::-1][:expand_k]
    return [neighbor_pages[i] for i in top_idx]


def compute_em(context: str, gold_answer: str) -> float:
    """Exact Match: 1.0 if gold answer appears case-insensitively in context."""
    return 1.0 if gold_answer.lower() in context.lower() else 0.0


print('All helper functions defined.')

All helper functions defined.


---
## Main Loop — Process Chunks 002–008

For each chunk we:
1. Load corpus + questions
2. Build FAISS index (embed all pages)
3. Build hyperlink graph (NetworkX)
4. For every question, run both Simple RAG and HyperRAG
5. Record graph-wins: Simple RAG EM=0 **and** HyperRAG EM=1
6. Free memory before the next chunk

⏱️ **Expected time:** ~5–15 min per chunk on CPU (embedding ~10k pages).  
The model is loaded once and reused across all chunks.

In [3]:
graph_wins_questions = []   # accumulated graph-wins question records
graph_wins_pages     = {}   # title → page dict (deduped across chunks)

for chunk_tag in CHUNKS_TO_PROCESS:
    corpus_path    = DATA_DIR / f'corpus_{chunk_tag}.json'
    questions_path = DATA_DIR / f'questions_{chunk_tag}.json'

    if not corpus_path.exists() or not questions_path.exists():
        print(f'[SKIP] chunk {chunk_tag} — files not found')
        continue

    # ── 1. Load data ──────────────────────────────────────────────────────────
    with open(corpus_path, encoding='utf-8')    as f: corpus    = json.load(f)
    with open(questions_path, encoding='utf-8') as f: questions = json.load(f)
    print(f'\n{"=" * 60}')
    print(f'Chunk {chunk_tag}: {len(corpus)} pages, {len(questions)} questions')

    # ── 2. Build FAISS index ──────────────────────────────────────────────────
    print(f'  Building FAISS index ...')
    index = build_index(corpus)
    print(f'  Index: {index.ntotal} vectors')

    # ── 3. Build hyperlink graph ──────────────────────────────────────────────
    print(f'  Building hyperlink graph ...')
    graph = build_graph(corpus)
    print(f'  Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges')

    title_to_page = {p['title']: p for p in corpus}

    # ── 4. Run both systems on every question ─────────────────────────────────
    chunk_wins = 0
    for q in tqdm(questions, desc=f'chunk {chunk_tag}', unit='q'):
        # Simple RAG
        simple_pages   = retrieve(q['question'], index, corpus, k=K)
        simple_context = '\n\n'.join(p['text'] for p in simple_pages)
        simple_em      = compute_em(simple_context, q['answer'])

        # If Simple RAG already found the answer, this is NOT a graph-win
        if simple_em == 1.0:
            continue

        # HyperRAG (only run if Simple RAG missed)
        expanded_pages = expand_neighbors(q['question'], simple_pages, corpus, graph, expand_k=EXPAND_K)
        all_pages      = simple_pages + expanded_pages
        hyper_context  = '\n\n'.join(p['text'] for p in all_pages)
        hyper_em       = compute_em(hyper_context, q['answer'])

        # Graph-win: Simple RAG missed (em=0), HyperRAG hit (em=1)
        if hyper_em == 1.0:
            chunk_wins += 1

            # Record the question with which chunk it came from
            graph_wins_questions.append({
                **q,
                'source_chunk':         chunk_tag,
                'simple_rag_em':        0.0,
                'hyperrag_em':          1.0,
                'hyperrag_titles':      [p['title'] for p in all_pages],
                'graph_expanded_titles': [p['title'] for p in expanded_pages],
            })

            # Collect corpus pages needed to answer this question:
            # supporting_titles (gold) + all retrieved pages (what HyperRAG used)
            needed_titles = set(q['supporting_titles']) | {p['title'] for p in all_pages}
            for title in needed_titles:
                if title not in graph_wins_pages and title in title_to_page:
                    graph_wins_pages[title] = title_to_page[title]

    print(f'  Graph-wins in chunk {chunk_tag}: {chunk_wins}')

    # ── 5. Free memory before next chunk ─────────────────────────────────────
    del index, graph, corpus, questions, title_to_page
    gc.collect()

print(f'\n{"=" * 60}')
print(f'TOTAL graph-wins questions : {len(graph_wins_questions)}')
print(f'TOTAL required corpus pages: {len(graph_wins_pages)}')


Chunk 002: 9999 pages, 5337 questions
  Building FAISS index ...
Loading sentence-transformers/all-MiniLM-L6-v2 ...
Model ready.


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

  Index: 9999 vectors
  Building hyperlink graph ...
  Graph: 9999 nodes, 80760 edges


chunk 002:   0%|          | 0/5337 [00:00<?, ?q/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

  Graph-wins in chunk 008: 221

TOTAL graph-wins questions : 2597
TOTAL required corpus pages: 15881


---
## Save Outputs

In [4]:
# questions-graph-wins.json — list of graph-wins question records
with open(OUT_QUESTIONS, 'w', encoding='utf-8') as f:
    json.dump(graph_wins_questions, f, indent=2, ensure_ascii=False)
print(f'Saved {len(graph_wins_questions)} questions → {OUT_QUESTIONS}')

# corpus-graph-wins.json — list of corpus pages (deduplicated)
corpus_list = list(graph_wins_pages.values())
with open(OUT_CORPUS, 'w', encoding='utf-8') as f:
    json.dump(corpus_list, f, indent=2, ensure_ascii=False)
print(f'Saved {len(corpus_list)} pages → {OUT_CORPUS}')

Saved 2597 questions → data\questions-graph-wins.json
Saved 15881 pages → data\corpus-graph-wins.json


---
## Summary & Inspection

In [5]:
from collections import Counter

# Per-chunk breakdown
chunk_counts = Counter(q['source_chunk'] for q in graph_wins_questions)
print('Graph-wins per chunk:')
for tag in CHUNKS_TO_PROCESS:
    print(f'  chunk {tag}: {chunk_counts.get(tag, 0)}')

# Question type breakdown
type_counts = Counter(q.get('type', 'unknown') for q in graph_wins_questions)
print(f'\nQuestion types: {dict(type_counts)}')

# Average number of graph-expanded pages that caused the win
avg_expanded = sum(len(q['graph_expanded_titles']) for q in graph_wins_questions) / max(len(graph_wins_questions), 1)
print(f'Avg expanded pages per win: {avg_expanded:.2f}')

# Preview first 5 graph-wins
print('\n--- First 5 graph-wins ---')
for q in graph_wins_questions[:5]:
    print(f'  [{q["source_chunk"]}] Q: {q["question"][:70]}')
    print(f'          Answer: {q["answer"]}')
    print(f'          Supporting pages: {q["supporting_titles"]}')
    print(f'          Expanded via graph: {q["graph_expanded_titles"]}')
    print()

Graph-wins per chunk:
  chunk 002: 384
  chunk 003: 382
  chunk 004: 410
  chunk 005: 389
  chunk 006: 420
  chunk 007: 391
  chunk 008: 221

Question types: {'bridge': 2597}
Avg expanded pages per win: 3.00

--- First 5 graph-wins ---
  [002] Q: What song by Jessi Colter did Erika Jo sing?
          Answer: "I'm Not Lisa"
          Supporting pages: ['Jessi Colter', 'Erika Jo']
          Expanded via graph: ['Like My Mother Does', 'Jessi Colter', 'Holly Dunn']

  [002] Q: What is the birthyear of the last born player to cross the Old Firm di
          Answer: 1979
          Supporting pages: ['Tom Dunbar (footballer)', 'Kenny Miller']
          Expanded via graph: ['Sam Allardyce', 'Martin Ødegaard', 'Riyad Mahrez']

  [002] Q: Under Cover is the ninth studio album by heavy metal vocalist Ozzy Osb
          Answer: Woman
          Supporting pages: ['Under Cover', 'Woman (John Lennon song)']
          Expanded via graph: ['How? (song)', 'Rob Halford', 'Electric Light Orchestra']

  [0

In [6]:
# Verify output files exist and have correct sizes
import os

for path in [OUT_QUESTIONS, OUT_CORPUS]:
    size_kb = os.path.getsize(path) / 1024
    data = json.load(open(path, encoding='utf-8'))
    print(f'{path.name}: {len(data)} records, {size_kb:.1f} KB')

questions-graph-wins.json: 2597 records, 1930.4 KB
corpus-graph-wins.json: 15881 records, 3445817.4 KB


---
## Merge: Build `final_corpus.json` and `final_questions.json`

Concatenate the chunk-001 baseline files with the graph-wins files, then deduplicate:

| Input A | Input B | Dedup key | Output |
|---|---|---|---|
| `corpus_001.json` | `corpus-graph-wins.json` | `title` | `final_corpus.json` |
| `questions_001.json` | `questions-graph-wins.json` | `id` | `final_questions.json` |

In [1]:
import json
from pathlib import Path

DATA_DIR = Path("data")  # adjust to "notebooks/data" if running from repo root

# ── Corpus merge ──────────────────────────────────────────────────────────────
corpus_a = json.loads((DATA_DIR / "corpus_001.json").read_text(encoding="utf-8"))
corpus_b = json.loads((DATA_DIR / "corpus-graph-wins.json").read_text(encoding="utf-8"))



In [2]:

seen_titles: set[str] = set()
final_corpus: list[dict] = []
for page in corpus_a + corpus_b:
    if page["title"] not in seen_titles:
        seen_titles.add(page["title"])
        final_corpus.append(page)

out_corpus = DATA_DIR / "final_corpus.json"
out_corpus.write_text(json.dumps(final_corpus, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"corpus_001        : {len(corpus_a):>6} pages")
print(f"corpus-graph-wins : {len(corpus_b):>6} pages")
print(f"final_corpus      : {len(final_corpus):>6} pages  (deduplicated by title)")
print(f"Saved → {out_corpus}")

print()



corpus_001        :   9999 pages
corpus-graph-wins :  15881 pages
final_corpus      :  24473 pages  (deduplicated by title)
Saved → data\final_corpus.json



In [3]:
# ── Questions merge ───────────────────────────────────────────────────────────
questions_a = json.loads((DATA_DIR / "questions_001.json").read_text(encoding="utf-8"))
questions_b = json.loads((DATA_DIR / "questions-graph-wins.json").read_text(encoding="utf-8"))

seen_ids: set[str] = set()
final_questions: list[dict] = []
for q in questions_a + questions_b:
    if q["id"] not in seen_ids:
        seen_ids.add(q["id"])
        final_questions.append(q)

out_questions = DATA_DIR / "final_questions.json"
out_questions.write_text(json.dumps(final_questions, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"questions_001        : {len(questions_a):>6} questions")
print(f"questions-graph-wins : {len(questions_b):>6} questions")
print(f"final_questions      : {len(final_questions):>6} questions  (deduplicated by id)")
print(f"Saved → {out_questions}")

questions_001        :   5329 questions
questions-graph-wins :   2597 questions
final_questions      :   7926 questions  (deduplicated by id)
Saved → data\final_questions.json
